In [0]:
import sys
from pathlib import Path

current = Path.cwd()

repo_root = next(
    (
        path
        for path in [current, *current.parents]
        if (
            path / "src" / "house_price_simple_analysis"
        ).is_dir()
    ),
    None,
)

if repo_root is None:
    raise FileNotFoundError(
        f"Project root not found from {current}"
    )

src_path = str(repo_root / "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

from house_price_simple_analysis.config import ProjectConfig

config = ProjectConfig.from_yaml(
    str(repo_root / "config" / "project_config.yml")
)

In [0]:
from house_price_simple_analysis.model_validator import ModelValidator

# Retrieve the URI generated by the training task
candidate_uri = dbutils.jobs.taskValues.get(
    taskKey="train_model",
    key="candidate_model_uri"
)

validator = ModelValidator(
    config=config,
    spark=spark,
    candidate_uri=candidate_uri,
    min_improvement=0.01
)

results = validator.validate() 

dbutils.jobs.taskValues.set(
    key="validation_evidence",
    value=results
)

print("Champion RMSE:", results["champion_rmse"])
print("Candidate RMSE:", results["candidate_rmse"])
print("Improvement:", f'{results["improvement"]:.2%}')
print("Approved:", results["approved"])

dbutils.jobs.taskValues.set(
    key="candidate_approved",
    value=results["approved"]
)